In [7]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../../.env")

from experiment0f.config import Experiment0fConfig
from experiment0f.benchmarks import load_all_tasks
from experiment0f.utils import make_client

config = Experiment0fConfig()
client = make_client(config)
tasks = load_all_tasks(n_per_dataset=100)

  [SWE-bench] 100 tasks loaded
  [BigCodeBench] 100 tasks loaded
  [GPQA] loaded from Wanfq/gpqa
  [GPQA] 100 tasks loaded
  Loaded 300 tasks (100 swe, 100 bcb, 100 gpqa)


In [8]:
# Summary and recommended max_tokens
max_completion = max(r["completion_tokens"] for r in results.values())
recommended = max_completion * 4

print("Completion tokens per dataset:")
for ds, r in results.items():
    print(f"  {ds:15s}: {r['completion_tokens']} tokens  ({r['response_chars']} chars)")

print(f"\nMax observed:        {max_completion} tokens")
print(f"Recommended limit:   {recommended} tokens  (4x max)")
print(f"\nAdd to config.py:")
print(f"  max_tokens_generation: int = {recommended}")

Completion tokens per dataset:
  swebench       : 927 tokens  (3591 chars)
  bigcodebench   : 1072 tokens  (4043 chars)
  gpqa           : 804 tokens  (1803 chars)

Max observed:        1072 tokens
Recommended limit:   4288 tokens  (4x max)

Add to config.py:
  max_tokens_generation: int = 4288


In [9]:
# Test GPT-5 with one BigCodeBench task
bcb_task = next(t for t in tasks if t["dataset"] == "bigcodebench")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": bcb_task["prompt"]},
]

try:
    resp = client.chat.completions.create(
        model="openai/gpt-5",
        messages=messages,
        temperature=0.6,
        max_tokens=2048,
    )
    text = resp.choices[0].message.content
    print(f"SUCCESS — openai/gpt-5 works via OpenRouter")
    print(f"  completion_tokens: {resp.usage.completion_tokens}")
    print(f"  response_chars:    {len(text)}")
    print(f"\nPreview:\n{text[:400]}")
except Exception as e:
    print(f"FAILED: {e}")

SUCCESS — openai/gpt-5 works via OpenRouter
  completion_tokens: 2048
  response_chars:    84

Preview:
import sqlite3
import random
def task_func(db_path,
          num_entries,
         


In [ ]:
# Call Opus 4.5 with no max_tokens on one task per dataset
results = {}

for ds, task in samples.items():
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": task["prompt"]},
    ]
    resp = client.chat.completions.create(
        model=config.evaluator_id,
        messages=messages,
        temperature=config.temperature,
    )
    text = resp.choices[0].message.content
    usage = resp.usage
    results[ds] = {
        "prompt_tokens":     usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens":      usage.total_tokens,
        "response_chars":    len(text),
        "response_preview":  text[:300],
    }
    print(f"\n[{ds}]")
    print(f"  prompt_tokens:     {usage.prompt_tokens}")
    print(f"  completion_tokens: {usage.completion_tokens}")
    print(f"  response_chars:    {len(text)}")
    print(f"  preview: {text[:200]}")

In [ ]:
# Sample one task from each dataset
samples = {}
for dataset in ["swebench", "bigcodebench", "gpqa"]:
    samples[dataset] = next(t for t in tasks if t["dataset"] == dataset)

for ds, task in samples.items():
    print(f"{ds}: {task['task_id']}")
    print(task["prompt"][:200])
    print("---")